<a href="https://colab.research.google.com/github/abhijadhav14/Data-Analytics-Using-Python/blob/main/end_to_end_data_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import random

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("CustomerChurnAnalytics") \
    .getOrCreate()
#Dataset
random.seed(42)
data = []
for _ in range(200):
    age = random.randint(18, 70)
    tenure = random.randint(1, 72)
    charges = round(random.uniform(20.0, 120.0), 2)
    contract = random.choice(["Month-to-month", "One year", "Two year"])


    churn_score = (charges / 120.0) - (tenure / 72.0)
    if contract == "Month-to-month":
        churn_score += 0.5

    churn = "Yes" if churn_score > 0.4 else "No"
    data.append((age, tenure, charges, contract, churn))

columns = ["Age", "TenureMonths", "MonthlyCharges", "ContractType", "Churn"]
df = spark.createDataFrame(data, columns)

# 3. Data Preprocessing
contract_indexer = StringIndexer(inputCol="ContractType", outputCol="ContractIndex")
df = contract_indexer.fit(df).transform(df)

churn_indexer = StringIndexer(inputCol="Churn", outputCol="label")
df = churn_indexer.fit(df).transform(df)

assembler = VectorAssembler(
    inputCols=["Age", "TenureMonths", "MonthlyCharges", "ContractIndex"],
    outputCol="features"
)
model_data = assembler.transform(df)

# 4. Split data (70% train, 30% test)
train_data, test_data = model_data.randomSplit([0.7, 0.3], seed=42)

# 5. Create & Train Classifier
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20)
model = rf.fit(train_data)

# 6. Predict on Test Data
predictions = model.transform(test_data)

# 7. Evaluate Model Accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print("--- Predictions vs Actuals (Top 10) ---")
predictions.select("TenureMonths", "MonthlyCharges", "ContractType", "Churn", "prediction").show(10)
print(f"Model Accuracy = {accuracy * 100:.2f}%")

spark.stop()

--- Predictions vs Actuals (Top 10) ---
+------------+--------------+--------------+-----+----------+
|TenureMonths|MonthlyCharges|  ContractType|Churn|prediction|
+------------+--------------+--------------+-----+----------+
|          42|         68.86|Month-to-month|  Yes|       0.0|
|          40|         56.47|Month-to-month|  Yes|       1.0|
|           6|         86.13|      One year|  Yes|       1.0|
|          20|         74.56|      One year|   No|       0.0|
|          71|         49.32|      Two year|   No|       0.0|
|           8|         60.26|      One year|   No|       0.0|
|          10|         73.76|      Two year|  Yes|       1.0|
|          38|         63.48|      One year|   No|       0.0|
|          25|         49.67|Month-to-month|  Yes|       1.0|
|          53|         68.56|Month-to-month|   No|       0.0|
+------------+--------------+--------------+-----+----------+
only showing top 10 rows
Model Accuracy = 96.88%
